In [1]:
import pandas as pd
import numpy as np

filtered_df = pd.read_csv(
    "../data/processed/filtered_netflix.csv"
)

print(filtered_df.shape)
filtered_df.head()

(4660641, 3)


,user_id,movie_id,rating
0,1488844,1,3
1,885013,1,4
2,30878,1,4
3,823519,1,3
4,893988,1,3


In [2]:
from surprise import Dataset, Reader, SVD
print("Success")

Success


In [3]:
svd_df = filtered_df.sample(
    500000,
    random_state=42
)

print(svd_df.shape)

(500000, 3)


In [4]:
from surprise import Dataset
from surprise import Reader
from surprise import SVD

In [5]:
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    svd_df[['user_id', 'movie_id', 'rating']],
    reader
)

In [6]:
svd_df.memory_usage(deep=True).sum() / (1024**2)

np.float64(15.2587890625)

In [7]:
from surprise import Dataset, Reader

reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    svd_df[['user_id', 'movie_id', 'rating']],
    reader
)

In [8]:
svd_df = filtered_df.sample(
    1000000,
    random_state=42
)

In [9]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [10]:
import time
from surprise import SVD

start = time.time()

model = SVD(
    n_factors=50,
    random_state=42
)

model.fit(trainset)

end = time.time()

print(
    f"Training Time: {end-start:.2f} seconds"
)

Training Time: 3.31 seconds


In [11]:
train_df_svd = pd.DataFrame(
    [
        (
            trainset.to_raw_uid(uid),
            trainset.to_raw_iid(iid),
            rating
        )
        for (uid, iid, rating)
        in trainset.all_ratings()
    ],
    columns=[
        "user_id",
        "movie_id",
        "rating"
    ]
)

train_df_svd["user_id"] = train_df_svd["user_id"].astype(int)
train_df_svd["movie_id"] = train_df_svd["movie_id"].astype(int)

In [12]:
from surprise import accuracy

predictions = model.test(testset)

rmse = accuracy.rmse(
    predictions,
    verbose=True
)

RMSE: 0.9972


In [13]:
all_movies = set(
    svd_df["movie_id"].unique()
)

def recommend_svd(user_id, top_n=10):

    watched = set(
        train_df_svd[
            train_df_svd["user_id"] == user_id
        ]["movie_id"]
    )

    candidates = all_movies - watched

    predictions = []

    for movie in candidates:

        pred = model.predict(
            user_id,
            movie
        )

        predictions.append(
            (movie, pred.est)
        )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]

In [14]:
sample_user = svd_df.iloc[0]['user_id']
recommend_svd(sample_user)

[(np.int64(33), 5),
 (np.int64(37), 5),
 (np.int64(85), 5),
 (np.int64(106), 5),
 (np.int64(165), 5),
 (np.int64(167), 5),
 (np.int64(175), 5),
 (np.int64(215), 5),
 (np.int64(223), 5),
 (np.int64(253), 5)]

In [15]:
def average_precision_at_k(
    recommended,
    relevant,
    k=10
):

    score = 0
    hits = 0

    for i, movie in enumerate(
        recommended[:k],
        start=1
    ):

        if movie in relevant:

            hits += 1
            score += hits / i

    if len(relevant) == 0:
        return 0

    return score / min(
        len(relevant),
        k
    )

In [16]:
test_df_svd = pd.DataFrame(
    testset,
    columns=[
        'user_id',
        'movie_id',
        'rating'
    ]
)

test_df_svd.head()

,user_id,movie_id,rating
0,36541,607,4.0
1,280380,571,3.0
2,2537976,788,4.0
3,1315249,831,3.0
4,2094968,996,5.0


In [17]:
sample_users = (
    test_df_svd['user_id']
    .drop_duplicates()
    .sample(
        100,
        random_state=42
    )
)

In [18]:
ap_scores = []

for user in sample_users:

    user_test = test_df_svd[
        test_df_svd['user_id'] == user
    ]

    relevant_movies = set(
        user_test[
            user_test['rating'] >= 4
        ]['movie_id']
    )

    if len(relevant_movies) == 0:
        continue

    recs = recommend_svd(
        user,
        top_n=10
    )

    recommended_movies = [
        movie
        for movie, _
        in recs
    ]

    ap = average_precision_at_k(
        recommended_movies,
        relevant_movies,
        k=10
    )
    ap_scores.append(ap)

In [19]:
import numpy as np
map10 = np.mean(ap_scores)

print("SVD MAP@10:",map10)

SVD MAP@10: 0.0010245901639344263
